# 05 - Pupil cognitive-load indices: LHIPA + RIPA2

**Audience:** computing pupil-based cognitive-load metrics on a
recording, and comparing them across phases / conditions.

Two metrics ship in Eye_lean:

- **LHIPA** (Duchowski et al. 2020, CHI '20): a wavelet-based count of
  modulus-maxima of the low/high-frequency pupil-coefficient ratio,
  normalized per second of pupil signal. Higher load ⇒ *lower* LHIPA.
- **RIPA2** (Jayawardena, Jayawardana & Gwizdka 2025, JEMR 18(6):70):
  the live cognitive-load metric written to the CSV `LiveLoadIndex`
  column during recording. RIPA2 is the difference between two
  Savitzky-Golay band-pass derivatives of the pupil signal, squared
  and clipped to [0, 1.5]. Higher RIPA2 ⇒ *higher* load.

Both Python implementations are byte-for-byte parity-matched against
the Unity on-device code (`RIPA2Analyzer.cs`,
`RealtimeCognitiveLoad.cs`), so you can recompute either offline on the
raw pupil column and get the same numbers.

**Requires** `PyWavelets` for LHIPA — install with
`pip install -e .[wavelet]`.

## What you'll get

- Cleaned pupil signal with blink masking.
- Whole-recording and sliding-window LHIPA + RIPA2.
- Per-phase LHIPA and per-phase RIPA2 against the four-phase
  `SampleExperiment` battery (FreeExploration, VisualSearch,
  CountingTask, ChangeDetection).

**On comparing phases.** Pupil-based load indices are most informative
when reported per condition / per phase: a single whole-session number
can hide opposing load swings (e.g. the participant works hard during
counting then rests during free exploration, averaging to "medium"
load). The session-pooled comparisons below are the canonical reporting
unit.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela
from eyelean_analysis.metrics.ripa2 import calculate_ripa2

ctx = ela.notebook_context()
print(ctx)

## 1. Pull pupil signal + handle blinks

Average the two eyes (skipping samples where both eyes are NaN), then
NaN-mask physiologically impossible diameters as a coarse blink filter.

In [ ]:
df = ctx.data.df
ts = ctx.data.get_timestamps()
if not ("left_pupil_diameter" in df.columns or "right_pupil_diameter" in df.columns):
    raise RuntimeError("No pupil columns in this CSV.")
left  = df["left_pupil_diameter"].to_numpy(dtype=float)  if "left_pupil_diameter"  in df.columns else None
right = df["right_pupil_diameter"].to_numpy(dtype=float) if "right_pupil_diameter" in df.columns else None

# Manual averaging with valid-count divisor — avoids "Mean of empty
# slice" RuntimeWarning that np.nanmean raises when both eyes are NaN.
if left is not None and right is not None:
    stacked = np.vstack([left, right])
    valid   = ~np.isnan(stacked)
    counts  = valid.sum(axis=0)
    safe    = np.where(valid, stacked, 0.0)
    with np.errstate(invalid='ignore', divide='ignore'):
        pupil = np.where(counts > 0, safe.sum(axis=0) / counts, np.nan)
elif left is not None:
    pupil = left
else:
    pupil = right

# Mask physiologically impossible diameters (sensor dropouts) to NaN.
mask = (pupil < 1.0) | (pupil > 9.0)
pupil_clean = pupil.copy()
pupil_clean[mask] = np.nan
print(f"Total samples: {len(pupil)}, NaN-masked: {int(mask.sum())} ({mask.mean()*100:.1f}%)")
sr = ctx.data.get_sample_rate() or 90.0
print(f"Sample rate:   {sr:.1f} Hz")

## 2. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(ts, pupil, color="silver", linewidth=0.6, label="raw")
ax.plot(ts, pupil_clean, color="steelblue", linewidth=0.7, label="masked")
ax.set_xlabel("time (s)")
ax.set_ylabel("pupil (mm)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Whole-recording LHIPA + RIPA2

A reference value over the whole recording. As noted above, this lumps
phases together — go to §5 for the per-phase split.

In [ ]:
lhipa_full = ela.calculate_lhipa(pupil_clean, sample_rate=sr)
ripa_full  = calculate_ripa2(pupil_clean, sample_rate=sr)
print(f"LHIPA (whole):  {lhipa_full.lhipa:.4f}   valid={lhipa_full.is_valid}   "
      f"n={lhipa_full.n_samples}")
print(f"RIPA2 (whole):  mean={float(np.nanmean(ripa_full.ripa2_raw)):.3f}   "
      f"max={float(np.nanmax(ripa_full.ripa2_raw)):.3f}   "
      f"n_valid={int(np.isfinite(ripa_full.ripa2_raw).sum())}")

## 4. Sliding-window LHIPA

Per-window load timeline. Pairs nicely with the `K(t)` plot in
notebook 04 for an integrated picture.

In [ ]:
win_s = 10.0
step_s = 5.0
win_n = int(win_s * sr)
step_n = int(step_s * sr)
centers, vals = [], []
i = 0
while i + win_n <= len(pupil_clean):
    r = ela.calculate_lhipa(pupil_clean[i:i + win_n], sample_rate=sr)
    centers.append(ts[i + win_n // 2])
    vals.append(r.lhipa if r.is_valid else np.nan)
    i += step_n

if not centers:
    print("Recording too short for sliding LHIPA.")
else:
    phase_colors = {
        "FreeExploration": "#1f77b4", "VisualSearch": "#ff7f0e",
        "CountingTask":    "#2ca02c", "ChangeDetection": "#d62728",
    }
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(centers, vals, marker="o", linewidth=1.0, color="steelblue")
    # Shade phase boundaries if available.
    if ctx.data.has_column("phase"):
        phase_col = df["phase"].to_numpy()
        cur, start = None, None
        for j in range(len(phase_col)):
            if phase_col[j] != cur:
                if cur in phase_colors:
                    ax.axvspan(start, ts[j], color=phase_colors[cur], alpha=0.10, zorder=0)
                cur, start = phase_col[j], ts[j]
        if cur in phase_colors:
            ax.axvspan(start, ts[-1], color=phase_colors[cur], alpha=0.10, zorder=0)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("LHIPA")
    ax.set_title(f"Sliding LHIPA  ({win_s:.0f}s window, {step_s:.0f}s step)")
    plt.tight_layout()
    plt.show()

## 5. Per-phase LHIPA and RIPA2

Slice the pupil signal by phase and run each metric on the resulting
sub-signal. Phases shorter than LHIPA's minimum input length (the
wavelet needs enough samples to decompose) report `valid=False`. RIPA2
runs on any phase ≥ a few seconds because it operates per-sample.

For both metrics the **direction of change between phases** is the
inferential quantity. The free-exploration phase functions as a
relatively low-demand baseline; task phases (VisualSearch, CountingTask,
ChangeDetection) impose external task demands and typically register
*lower* LHIPA and *higher* RIPA2 relative to free exploration.

In [ ]:
PHASES = ("FreeExploration", "VisualSearch", "CountingTask", "ChangeDetection")
phase_col = df["phase"].to_numpy() if ctx.data.has_column("phase") else None
if phase_col is None:
    print("This recording has no phase column; per-phase split not applicable.")
    per_phase_pupil = None
else:
    rows = []
    for ph in PHASES:
        mask_ph = phase_col == ph
        n = int(mask_ph.sum())
        if n < int(5 * sr):   # need >=5s for either metric
            rows.append({"phase": ph, "duration_s": round(n / sr, 1),
                          "lhipa": None, "ripa2_mean": None, "ripa2_max": None,
                          "note": "phase too short"})
            continue
        pupil_ph = pupil_clean[mask_ph]
        lh = ela.calculate_lhipa(pupil_ph, sample_rate=sr)
        rp = calculate_ripa2(pupil_ph, sample_rate=sr)
        rows.append({
            "phase":        ph,
            "duration_s":   round(n / sr, 1),
            "lhipa":        round(float(lh.lhipa), 4) if lh.is_valid else None,
            "ripa2_mean":   round(float(np.nanmean(rp.ripa2_raw)), 3),
            "ripa2_max":    round(float(np.nanmax(rp.ripa2_raw)), 3),
            "note":         "" if lh.is_valid else (lh.error_message or "LHIPA invalid"),
        })
    per_phase_pupil = pd.DataFrame(rows)
    display(per_phase_pupil)

In [ ]:
if per_phase_pupil is None or per_phase_pupil["lhipa"].isna().all():
    print("Per-phase plot skipped — no phases produced valid LHIPA.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    valid = per_phase_pupil.dropna(subset=["lhipa"]).reset_index(drop=True)

    axes[0].bar(valid["phase"], valid["lhipa"],
                color=[palette[i] for i in range(len(valid))])
    axes[0].set_ylabel("LHIPA  (lower = higher load)")
    axes[0].set_title("Per-phase LHIPA")
    for i, v in enumerate(valid["lhipa"]):
        axes[0].text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

    axes[1].bar(valid["phase"], valid["ripa2_mean"],
                color=[palette[i] for i in range(len(valid))])
    axes[1].set_ylabel("RIPA2  (higher = higher load)")
    axes[1].set_title("Per-phase RIPA2 (mean)")
    for i, v in enumerate(valid["ripa2_mean"]):
        axes[1].text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

    for ax in axes:
        ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    plt.show()

## 6. Relative-to-FreeExploration deltas

The standard reporting unit for load is the delta between a task and a
matched low-demand reference. We take FreeExploration as the in-session
baseline and report each task phase as `(task − baseline)`. Positive
RIPA2 delta = the task imposed more load; negative LHIPA delta = same.

In [ ]:
if per_phase_pupil is None or "FreeExploration" not in per_phase_pupil["phase"].values:
    print("FreeExploration baseline not present — skipping delta table.")
else:
    base_row = per_phase_pupil[per_phase_pupil["phase"] == "FreeExploration"].iloc[0]
    base_lhipa = base_row["lhipa"]
    base_ripa  = base_row["ripa2_mean"]
    deltas = []
    for _, r in per_phase_pupil.iterrows():
        if r["phase"] == "FreeExploration":
            continue
        deltas.append({
            "phase":           r["phase"],
            "lhipa":           r["lhipa"],
            "delta_lhipa":     None if (r["lhipa"] is None or base_lhipa is None) else round(r["lhipa"] - base_lhipa, 4),
            "ripa2_mean":      r["ripa2_mean"],
            "delta_ripa2":     None if (r["ripa2_mean"] is None or base_ripa is None) else round(r["ripa2_mean"] - base_ripa, 3),
        })
    print(f"Baseline (FreeExploration): LHIPA={base_lhipa}  RIPA2_mean={base_ripa}")
    display(pd.DataFrame(deltas))

## What next

- Notebook `06_sample_experiment` produces a tidy per-phase DataFrame
  with LHIPA + fixation + scanpath + entropy in one call via
  `analyze_sample_experiment()`.
- Notebook `04_eye_movements` does the analogous per-phase split for
  K-coefficient and fixation/saccade statistics.